<style>
.jp-Notebook, .notebook-container, .markdown-body {font-family: Arial, Helvetica, sans-serif;}
.jp-MarkdownOutput, .text_cell_render {font-size: 18px; line-height: 1.65;}
h1 {font-size: 2.25rem !important; margin-top: 0.35em !important;}
h2 {font-size: 1.65rem !important; margin-top: 1.35em !important;}
h3 {font-size: 1.25rem !important; margin-top: 1.1em !important;}
table {font-size: 0.95em;}
blockquote {border-left: 4px solid #aaa; padding-left: 1rem;}
</style>


# 06 · Obtener datos desde web y APIs

<p><a href="https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/content/statistics-module-rework/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/notebooks/06_Web_Scraping_y_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a></p>

> **Objetivo:** entender dos rutas frecuentes para obtener datos externos: HTML (scraping) y APIs, y convertir lo obtenido en información que pueda analizarse.

> **Nota:** esta notebook es un **puente práctico**, no un tema central de estadística.


<img src="https://raw.githubusercontent.com/mauriciorslrv/DS_basics/content/statistics-module-rework/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/imgs/Web_scrapping.png" alt="Web scraping" width="760">


## 1. Dos caminos

**Web:** petición → HTML → localizar → extraer → validar → estructurar  
**API:** petición → JSON/XML → validar → transformar

Cuando existe una API oficial adecuada, suele ser más estable que depender de la estructura visual de una página.


<img src="https://raw.githubusercontent.com/mauriciorslrv/DS_basics/content/statistics-module-rework/02%20An%C3%A1lisis%20Estad%C3%ADstico%20de%20Datos/imgs/Elementos_Web_scrap.png" alt="Elementos de web scraping" width="800">


## 2. Primero: aprender el parser sin depender de internet

### ¿Por qué?
Un sitio real puede cambiar, estar temporalmente caído o bloquear una petición. Para aprender la lógica de selectores, primero usamos HTML local y comprobamos que el parser funciona.


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

html = '''
<html><body>
<div class="producto" data-stock="12"><h2>Café</h2><span class="precio">120</span></div>
<div class="producto" data-stock="5"><h2>Té</h2><span class="precio">85</span></div>
<div class="producto" data-stock="8"><h2>Cacao</h2><span class="precio">150</span></div>
<div class="producto" data-stock="3"><h2>Infusión</h2><span class="precio">70</span></div>
<div class="producto" data-stock="15"><h2>Chocolate</h2><span class="precio">135</span></div>
</body></html>
'''

soup=BeautifulSoup(html,"html.parser")

filas=[]
for producto in soup.select(".producto"):
    filas.append({
        "producto":producto.select_one("h2").get_text(strip=True),
        "precio":float(producto.select_one(".precio").get_text(strip=True)),
        "stock":int(producto.get("data-stock"))
    })

df_local=pd.DataFrame(filas)
df_local


### 💡 IDEA
Scrapear no es "copiar una página": es convertir una estructura semiestructurada en **datos verificables**.

Si cambia el HTML, el parser puede romperse. Por eso conviene separar:

**obtener página → parsear → validar → analizar**.


## 3. Ejercicio real y más completo: Books to Scrape

Usaremos `books.toscrape.com`, un sitio **sandbox creado específicamente para practicar scraping**.

### ¿Qué vamos a hacer?
1. descargar varias páginas;
2. extraer título, precio, rating y disponibilidad;
3. construir un DataFrame;
4. validar que los campos tengan sentido;
5. crear gráficas;
6. aplicar una regla de decisión didáctica.

> Los precios y ratings del sitio son ficticios/aleatorios; la decisión es un ejercicio de análisis, no una recomendación comercial real.


In [ ]:
import requests
from urllib.parse import urljoin

BASE_URL="https://books.toscrape.com/"
RATING_MAP={"One":1,"Two":2,"Three":3,"Four":4,"Five":5}

def parsear_libros(html_text, source_url):
    soup=BeautifulSoup(html_text,"html.parser")
    filas=[]

    for tarjeta in soup.select("article.product_pod"):
        enlace=tarjeta.select_one("h3 a")
        precio_txt=tarjeta.select_one(".price_color").get_text(strip=True)
        rating_tag=tarjeta.select_one("p.star-rating")
        disponibilidad=tarjeta.select_one(".availability").get_text(" ",strip=True)

        rating_word=next(
            (clase for clase in rating_tag.get("class",[]) if clase in RATING_MAP),
            None
        )

        precio=float(
            precio_txt
            .replace("£","")
            .replace("Â","")
            .strip()
        )

        filas.append({
            "titulo":enlace.get("title", enlace.get_text(strip=True)),
            "precio_gbp":precio,
            "rating":RATING_MAP.get(rating_word, np.nan),
            "en_stock":"In stock" in disponibilidad,
            "url":urljoin(source_url,enlace.get("href"))
        })

    return pd.DataFrame(filas)


### 3.1 Verificación del parser con HTML controlado

Antes de depender de internet, probamos el mismo parser con una estructura equivalente. Si esta celda falla, el problema está en nuestra lógica, no en la conexión.


In [ ]:
html_prueba = '''
<html><body>
<article class="product_pod">
  <h3><a href="catalogue/libro-a_1/index.html" title="Libro A">Libro A</a></h3>
  <p class="price_color">£18.50</p>
  <p class="star-rating Four"></p>
  <p class="availability">In stock</p>
</article>
<article class="product_pod">
  <h3><a href="catalogue/libro-b_2/index.html" title="Libro B">Libro B</a></h3>
  <p class="price_color">£42.00</p>
  <p class="star-rating Two"></p>
  <p class="availability">In stock</p>
</article>
</body></html>
'''

test_parser=parsear_libros(html_prueba, BASE_URL)

assert len(test_parser)==2
assert test_parser["rating"].tolist()==[4,2]
assert np.allclose(test_parser["precio_gbp"],[18.5,42.0])

test_parser


### 3.2 Descargar varias páginas

Por defecto tomamos 5 páginas (100 libros). Puedes aumentar `paginas` para ampliar el conjunto.

Incluimos manejo de errores y un pequeño fallback para que el resto de la notebook siga siendo ejecutable si la red no está disponible.


In [ ]:
def scrapear_catalogo(paginas=5, timeout=15):
    frames=[]

    for pagina in range(1,paginas+1):
        url=f"https://books.toscrape.com/catalogue/page-{pagina}.html"

        response=requests.get(
            url,
            timeout=timeout,
            headers={"User-Agent":"Mozilla/5.0 (educational statistics notebook)"}
        )
        response.raise_for_status()

        df_pagina=parsear_libros(response.text,url)

        if df_pagina.empty:
            raise ValueError(f"No se encontraron libros en {url}")

        df_pagina["pagina"]=pagina
        frames.append(df_pagina)

    return pd.concat(frames,ignore_index=True)

try:
    libros=scrapear_catalogo(paginas=5)
    fuente="web en vivo"
except requests.RequestException as exc:
    print("No fue posible acceder al sitio en esta ejecución:", exc)
    print("Usamos un conjunto local de respaldo para continuar con el análisis.")
    libros=pd.concat([test_parser]*6,ignore_index=True)
    libros["pagina"]=np.repeat(np.arange(1,7),len(test_parser))
    fuente="fallback local"

print("Fuente:", fuente)
print("Filas:", len(libros))
libros.head()


## 4. Validar antes de analizar

### ¿Por qué?
El scraping puede "funcionar" técnicamente y aun así extraer datos incorrectos.

Revisamos:

- número de filas;
- faltantes;
- tipos;
- rangos plausibles;
- duplicados;
- cobertura de páginas.


In [ ]:
validacion=pd.Series({
    "filas":len(libros),
    "titulos_unicos":libros["titulo"].nunique(),
    "precios_faltantes":libros["precio_gbp"].isna().sum(),
    "ratings_faltantes":libros["rating"].isna().sum(),
    "precio_min":libros["precio_gbp"].min(),
    "precio_max":libros["precio_gbp"].max()
})
validacion


## 5. De datos a gráficas

### ¿Por qué?
La extracción es sólo el inicio. El objetivo del pipeline es llegar a una descripción que nos ayude a formular o responder preguntas.


In [ ]:
fig, ax=plt.subplots(figsize=(8,5))
ax.hist(libros["precio_gbp"],bins=15)
ax.set_xlabel("Precio (£)")
ax.set_ylabel("Número de libros")
ax.set_title("Distribución de precios")
plt.show()


In [ ]:
precio_por_rating=libros.groupby("rating",dropna=False)["precio_gbp"].agg(["count","mean","median"])
precio_por_rating


In [ ]:
fig, ax=plt.subplots(figsize=(8,5))
precio_por_rating["mean"].plot(kind="bar",ax=ax)
ax.set_xlabel("Rating")
ax.set_ylabel("Precio medio (£)")
ax.set_title("Precio medio por rating")
plt.show()


## 6. De gráficas a una decisión explícita

Imagina una selección didáctica con dos reglas:

- rating de **4 o 5 estrellas**;
- precio de **£30 o menos**.

No estamos diciendo que ésos sean libros "mejores": sólo convertimos criterios explícitos en una regla reproducible.


In [ ]:
shortlist=(
    libros
    .query("rating >= 4 and precio_gbp <= 30")
    .sort_values(["rating","precio_gbp"],ascending=[False,True])
)

print(f"Libros que cumplen la regla: {len(shortlist)} de {len(libros)}")
shortlist[["titulo","precio_gbp","rating"]].head(10)


### Interpretación y decisión

Una buena decisión analítica debería documentar:

1. **qué datos se usaron**;
2. **qué regla o métrica define la decisión**;
3. **qué limitaciones tiene la fuente**;
4. **qué cambiaría la conclusión**.

En este sandbox, precios y ratings son ficticios. Por tanto, la conclusión válida es metodológica:

> **podemos construir un pipeline reproducible desde una web hasta una regla de decisión; no podemos inferir preferencias reales del mercado a partir de estos datos ficticios.**


## 7. APIs: datos estructurados

Una API suele entregar datos ya estructurados. El patrón básico es:

**request → status → JSON → normalizar → validar → DataFrame**


In [ ]:
respuesta_json={
    "ciudad":"Ejemplo",
    "temperatura":22.4,
    "humedad":0.48
}

pd.DataFrame([respuesta_json])


In [ ]:
# Patrón de una API real:
#
# response = requests.get("https://api.example.com/recurso", timeout=10)
# response.raise_for_status()
# data = response.json()
# df = pd.json_normalize(data)
#
# Después: validar campos, tipos, unidades y cobertura.


## 8. Ética y diseño

- revisa términos de servicio y `robots.txt` cuando aplique;
- prefiere APIs oficiales cuando exista una opción adecuada;
- evita recolectar datos personales sin base legítima;
- no sobrecargues servidores;
- respeta rate limits;
- documenta fuente y fecha;
- asume que una página web puede cambiar.

### ⚠️ ERROR ÚTIL
Que algo sea técnicamente accesible no significa que sea apropiado recolectarlo o reutilizarlo.


## 9. Mini reto

Amplía el ejercicio de Books to Scrape:

1. toma 10 páginas;
2. crea una variable de rango de precio (`bajo`, `medio`, `alto`);
3. compara rating por rango;
4. define una regla de selección distinta;
5. escribe una decisión y una limitación.

Como extensión, conecta una API pública y combina una variable de la API con un DataFrame propio.


## 📚 Material adicional
- [Requests · Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) — peticiones HTTP, errores y parámetros.
- [Beautiful Soup documentation](https://beautiful-soup-4.readthedocs.io/en/latest/) — selectores y navegación del HTML.
- [MDN · HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP) — conceptos de HTTP.
- [Books to Scrape](https://books.toscrape.com/) — sitio sandbox creado específicamente para practicar scraping.

### 🧾 Términos clave
**HTTP · status code · HTML · selector CSS · parser · paginación · JSON · API · rate limit · robots.txt**


## Cierre del módulo

**pregunta → describir → modelar incertidumbre → contrastar → simular → obtener nuevas fuentes → decidir**

La base queda lista para entrar a Machine Learning sin tratar los modelos como cajas negras.
